In [1]:
%%capture
!pip install -q transformers accelerate bitsandbytes datasets peft trl==0.12.2

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
   

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
model_name = "Qwen/Qwen2-7B-Instruct"
max_seq_length = 512

# QLoRA: load model in 4-bit to fit on T4 (16GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
print(f"Model loaded: {model_name}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen2-7B-Instruct
GPU memory used: 5.60 GB


In [5]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


In [6]:
# Using Dolly-15k - a solid instruction-following dataset
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
dataset = dataset.select(range(500))  # 500 examples to keep training fast on T4
print(f"Dataset size: {len(dataset)} examples")
print(f"Sample keys: {list(dataset[0].keys())}")
print(f"Sample instruction: {dataset[0]['instruction'][:100]}...")

README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Dataset size: 500 examples
Sample keys: ['instruction', 'context', 'response', 'category']
Sample instruction: When did Virgin Australia start operating?...


In [7]:
def formatting_func(example):
    text = f"""### Instruction:
{example["instruction"]}

### Input:
{example["context"]}

### Response:
{example["response"]}"""
    return [text]

In [8]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    formatting_func = formatting_func,
    max_seq_length = max_seq_length,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        save_strategy = "no",
        gradient_checkpointing = True,
    ),
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
trainer.train()
print("Training complete!")

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key 

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss


Training complete!


In [10]:
# Save the LoRA adapter
trainer.model.save_pretrained("qwen_lora_adapter")
tokenizer.save_pretrained("qwen_lora_adapter")
print("Adapter saved to qwen_lora_adapter/")

Adapter saved to qwen_lora_adapter/


In [11]:
# Quick inference test
model.eval()

prompt = """### Instruction:
What is the capital of France?

### Input:


### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


### Instruction:
What is the capital of France?

### Input:


### Response:
The capital of France is Paris.


In [12]:
!zip -r qwen_lora_adapter.zip qwen_lora_adapter

  adding: qwen_lora_adapter/ (stored 0%)
  adding: qwen_lora_adapter/added_tokens.json (deflated 36%)
  adding: qwen_lora_adapter/merges.txt (deflated 57%)
  adding: qwen_lora_adapter/adapter_config.json (deflated 58%)
  adding: qwen_lora_adapter/tokenizer.json (deflated 81%)
  adding: qwen_lora_adapter/special_tokens_map.json (deflated 46%)
  adding: qwen_lora_adapter/tokenizer_config.json (deflated 65%)
  adding: qwen_lora_adapter/README.md (deflated 65%)
  adding: qwen_lora_adapter/vocab.json (deflated 61%)
  adding: qwen_lora_adapter/adapter_model.safetensors (deflated 59%)


In [16]:
from google.colab import files
files.download("qwen_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>